In [ ]:
import sys
import subprocess

def install_gnn_dependencies():
    # Detect Python version
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    print(f"Detected Python {version}")

    # Map Python versions to compatible Torch/PyG versions
    if version == "3.12":
        torch_ver = "2.2.0"
    else:
        torch_ver = "2.1.0"
    
    # Define the index URL for the specific binaries
    pyg_index = f"https://data.pyg.org/whl/torch-{torch_ver}+cpu.html"
    torch_index = "https://download.pytorch.org/whl/cpu"

    print(f"Executing prioritized installation for Torch {torch_ver}...")

    try:
        # Step 1: Install Torch FIRST and ALONE
        # This ensures the foundation is solid before anything else is added
        print("Step 1: Prioritizing Torch installation...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user",
                              f"torch=={torch_ver}", "--extra-index-url", torch_index])

        # Step 2: Install NumPy (resolves the "Numpy is not available" runtime error)
        print("Step 2: Installing NumPy 1.26.4...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--upgrade",
                              "numpy==1.26.4"])

        # Step 3: Install Scatter and Sparse Binaries with --user flag
        # These REQUIRE torch to be fully installed to see the headers
        print("Step 3: Installing GNN Extensions (Scatter/Sparse)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user",
                              "torch-scatter", "torch-sparse", "-f", pyg_index])

        # Step 4: Install remaining project requirements
        print("Step 4: Installing remaining requirements...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", 
                              "-r", "requirements.txt"])
        
        print("\n✓ Environment successfully configured with Torch prioritized.")
        
    except subprocess.CalledProcessError as e:
        print(f"\n× Installation failed at step {e.cmd} with error code {e.returncode}.")
        print("Ensure you have internet access and sufficient disk quota.")

install_gnn_dependencies()

Detected Python 3.10
Installing Foundations (NumPy/Torch 2.1.0) and matching GNN extensions...
Step 1: Installing NumPy 1.26.4 and Torch...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu
Step 2: Installing GNN Extensions (Scatter/Sparse)...
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cpu.html
Step 3: Installing remaining requirements...
Ignoring torch: markers 'python_version >= "3.12"' don't match your environment
Ignoring torch-scatter: markers 'python_version >= "3.12"' don't match your environment
Ignoring torch-sparse: markers 'python_version >= "3.12"' don't match your environment
  Using cached torch_sparse-0.6.17.tar.gz (209 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'

× Installation failed with error code 1.
Try running the commands manually in the terminal 

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [17 lines of output]
      Traceback (most recent call last):
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
        File "/student/minalex/.local/lib/python3.10/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/tmp/pip-build-env-pddp1ub6/overlay/local/lib/python3.10/dist-packages/setuptools/build_meta.py", line 333, in get_requires_for_build_wheel
          return self._get_build_requires(config_settings, requirements=[])
    

# Download banksim dataset from kaggle

**Citation:** Lopez-Rojas, Edgar Alonso; Axelsson, Stefan. *Banksim: A bank payments simulator for fraud detection research*. Inproceedings 26th European Modeling and Simulation Symposium, EMSS 2014, Bordeaux, France, pp. 144–152, Dime University of Genoa, 2014, ISBN: 9788897999324. https://www.researchgate.net/publication/265736405_BankSim_A_Bank_Payment_Simulation_for_Fraud_Detection_Research

In [3]:
import kagglehub
import os
import shutil
from csv_to_gzip import csv_to_gzip

# Check for required files in banksim directory
files_to_check = [
    "banksim/bs140513_032310.csv.gz",
    "banksim/bsNET140513_032310.csv.gz"
]

files_exist = all(os.path.exists(f) for f in files_to_check)

if files_exist:
    print("Both required files found. Skipping download.")
else:
    print("Required files missing. Downloading dataset...")
    path = kagglehub.dataset_download("ealaxi/banksim1")
    shutil.copytree(path, "banksim", dirs_exist_ok=True)
    print("Downloaded dataset to 'banksim'.")
    
    # Compress CSV files to gzip
    for csv_file in ['bs140513_032310.csv', 'bsNET140513_032310.csv']:
        csv_path = os.path.join("banksim", csv_file)
        if os.path.exists(csv_path):
            csv_to_gzip(csv_path)
            print(f"Compressed {csv_file} to {csv_file}.gz")
            os.remove(csv_path)

print("Path to dataset files:", "banksim")

Required files missing. Downloading dataset...
Downloaded dataset to 'banksim'.
Compressing: banksim/bs140513_032310.csv
Output: banksim/bs140513_032310.csv.gz
Original size: 48,986,035 bytes
Compressed size: 7,075,076 bytes
Compression ratio: 85.6%
Successfully created: banksim/bs140513_032310.csv.gz
Compressed bs140513_032310.csv to bs140513_032310.csv.gz
Compressing: banksim/bsNET140513_032310.csv
Output: banksim/bsNET140513_032310.csv.gz
Original size: 32,665,953 bytes
Compressed size: 6,219,961 bytes
Compression ratio: 81.0%
Successfully created: banksim/bsNET140513_032310.csv.gz
Compressed bsNET140513_032310.csv to bsNET140513_032310.csv.gz
Path to dataset files: banksim


# Download the scotiabank data files

In [4]:
import gdown
import zipfile
import os
from csv_to_gzip import csv_to_gzip

data_path = "data"
# Check for required files in data directory
files_to_check = [
    "data/labels.csv.gz",
    "data/kyc_individual.csv.gz",
    "data/kyc_smallbusiness.csv.gz",
    "data/card.csv.gz"
]

files_exist = all(os.path.exists(f) for f in files_to_check)
files_exist = files_exist and os.path.isdir(data_path) and len(os.listdir(data_path)) == 12

if files_exist:
    print("Required files found. Skipping download.")
else:
    print("Required files missing. Downloading dataset...")
    
    # Download data.zip
    url = "https://drive.google.com/file/d/1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG/view?usp=sharing"
    zip_path = "data.zip"

    print("Downloading data.zip...")
    gdown.download(url, zip_path, fuzzy=True)
    print("Download complete.")

    # Extract the zip file
    print("Extracting data.zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("data")
    print("Extraction complete.")

    # Compress all CSV files in the data directory to .csv.gz
    print("Compressing CSV files to gzip...")
    for filename in os.listdir(data_path):
        if filename.endswith('.csv'):
            csv_path = os.path.join(data_path, filename)
            csv_to_gzip(csv_path)
            print(f"Compressed {filename} to {filename}.gz")
            os.remove(csv_path)

    # Clean up the zip file
    os.remove(zip_path)
    print("Cleaned up data.zip")

print("Data files ready in 'data' directory")


Required files missing. Downloading dataset...


Downloading...
From (original): https://drive.google.com/uc?id=1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG
From (redirected): https://drive.google.com/uc?id=1A7w3GqZTCVsv-A8gXj5NKV2FNc0zoduG&confirm=t&uuid=d2829247-df6d-416e-9945-e7a76a2c5648
To: /student/minalex/imi-bigdata-2026/data.zip
100%|██████████| 114M/114M [00:01<00:00, 60.0MB/s] 


Download complete.
Extracting data.zip...
Extraction complete.
Compressing CSV files to gzip...
Compressing: data/kyc_individual.csv
Output: data/kyc_individual.csv.gz
Original size: 4,035,647 bytes
Compressed size: 975,576 bytes
Compression ratio: 75.8%
Successfully created: data/kyc_individual.csv.gz
Compressed kyc_individual.csv to kyc_individual.csv.gz
Compressing: data/kyc_occupation_codes.csv
Output: data/kyc_occupation_codes.csv.gz
Original size: 4,268 bytes
Compressed size: 1,733 bytes
Compression ratio: 59.4%
Successfully created: data/kyc_occupation_codes.csv.gz
Compressed kyc_occupation_codes.csv to kyc_occupation_codes.csv.gz
Compressing: data/cheque.csv
Output: data/cheque.csv.gz
Original size: 12,558,384 bytes
Compressed size: 3,079,795 bytes
Compression ratio: 75.5%
Successfully created: data/cheque.csv.gz
Compressed cheque.csv to cheque.csv.gz
Compressing: data/abm.csv
Output: data/abm.csv.gz
Original size: 13,889,321 bytes
Compressed size: 3,379,367 bytes
Compression r

# GraphSAGE — Inductive Fraud Detection

### Design Choices for Inductive Learning

Rather than passing edge features into the convolution (GAT-style), we **pre-aggregate transaction features into the customer node** itself. This means any new customer node can be fully described using only their own KYC data + a few transaction aggregates — no need to re-embed the whole graph.

**Customer node (13 features):**
- KYC: `age, income, tenure, sales, emp_count, is_biz`
- Transaction aggregates: `avg_amount, max_amount, std_amount, txn_count, cash_rate, ecom_rate, velocity_24h`

**Category / City nodes:** learned embeddings (initialized to one-hot, updated during training)

**Graph edges:** customer → category, customer → city (bidirectional via `ToUndirected`)


# Prepare transaction nodes for Graph Attention Network

Combine the BankSim and Scotiabank transactions into a unified master pool aligned to the card schema:

**Schema**
[transaction_id, customer_id, amount_cad, debit_credit, transaction_datetime, merchant_category, ecommerce_ind, cash_indicator, country, province, city, is_fraud, source_dataset, time_delta, velocity_24h, cust_idx, cat_idx, city_idx]

**Field definitions**
- **transaction_id**: Unique transaction identifier (BankSim uses `BS_{i}`).
- **customer_id**: Unique customer identifier.
- **amount_cad**: Transaction amount in CAD (BankSim amounts multiplied by 1.62 from EUR).
- **debit_credit**: Debit/credit flag (BankSim uses `debit`; Scotiabank uses source values).
- **transaction_datetime**: Transaction timestamp (BankSim derived from `step`).
- **merchant_category**: Mapped industry category or transfer type.
- **ecommerce_ind**: 1 if e-commerce (e.g., tech/content), else 0.
- **cash_indicator**: 1 for cash-like activity (ABM), else 0.
- **country/province/city**: Location fields (BankSim defaults to Canada/Ontario/Toronto; transfers use N/A).
- **is_fraud**: Fraud label (BankSim from `fraud`; Scotiabank from `labels`, default -1 if unknown).
- **source_dataset**: Origin dataset (banksim, scotia_card, scotia_abm, scotia_transfer).
- **time_delta**: Minutes since the customer’s previous transaction (0 for first transaction).
- **velocity_24h**: Count of customer transactions in the trailing 24 hours (including current).
- **cust_idx/cat_idx/city_idx**: Integer-encoded indices for graph nodes.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import gc

banksim_path = "banksim"
data_path = "data"

def get_mapping(series):
    unique_vals = series.unique()
    return {val: i for i, val in enumerate(unique_vals)}, len(unique_vals)

def build_master_transaction_pool(banksim_path, data_path):
    def shrink_and_categorize(df):
        for col in df.select_dtypes(include=['object']):
            df[col] = df[col].astype('category')
        for col in df.select_dtypes(include=['float']):
            df[col] = pd.to_numeric(df[col], downcast='float')
        for col in df.select_dtypes(include=['integer']):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        return df

    print("Loading Banksim...")
    bs = pd.read_csv(os.path.join(banksim_path, 'bs140513_032310.csv.gz'))
    bs.columns = bs.columns.str.replace("'", "")
    for col in ['customer', 'category']:
        bs[col] = bs[col].str.replace("'", "").astype('category')

    start_date = datetime(year=2024, month=11, day=1)
    bs_master = pd.DataFrame({
        'transaction_id': np.arange(len(bs), dtype='int32'),
        'customer_id': bs['customer'],
        'amount_cad': (bs['amount'] * 1.62).astype('float32'),
        'debit_credit': 'debit',
        'transaction_datetime': bs['step'].apply(lambda x: start_date + timedelta(days=x)),
        'merchant_category': bs['category'],
        'ecommerce_ind': np.where(bs['category'].isin(['es_tech', 'es_contents']), 1, 0).astype('int8'),
        'cash_indicator': np.int8(0),
        'country': 'Canada', 'province': 'Ontario', 'city': 'Toronto',
        'is_fraud': bs['fraud'].astype('int8'),
        'source_dataset': 'banksim'
    })
    del bs; gc.collect()

    labels_df = pd.read_csv(os.path.join(data_path, 'labels.csv.gz'))
    labels_df['customer_id'] = labels_df['customer_id'].astype('category')

    def process_scotia(name, filename):
        print(f"  Processing {name}...")
        df = pd.read_csv(os.path.join(data_path, filename))
        df['customer_id'] = df['customer_id'].astype('category')
        df = df.merge(labels_df[['customer_id', 'label']], on='customer_id', how='left')
        df['is_fraud'] = df['label'].fillna(-1).astype('int8')
        df['source_dataset'] = name
        df.drop(columns=['label'], inplace=True)
        return shrink_and_categorize(df)

    card_master = process_scotia('scotia_card', 'card.csv.gz')
    card_master['cash_indicator'] = np.int8(0)

    abm_master = process_scotia('scotia_abm', 'abm.csv.gz')
    abm_master['merchant_category'] = 'ATM_WITHDRAWAL'
    abm_master['ecommerce_ind'] = np.int8(0)
    abm_master['cash_indicator'] = np.int8(1)

    others_list = []
    for name in ['cheque', 'eft', 'emt', 'westernunion', 'wire']:
        tmp = process_scotia(f'scotia_{name}', f'{name}.csv.gz')
        tmp['merchant_category'] = name.upper()
        others_list.append(tmp)

    others = pd.concat(others_list, axis=0)
    others['country'], others['province'], others['city'] = 'Canada', 'N/A', 'N/A'
    others['ecommerce_ind'] = np.int8(0)
    others['cash_indicator'] = np.int8(0)
    del others_list, labels_df; gc.collect()

    print("Final merge...")
    master_pool = pd.concat([bs_master, card_master, abm_master, others], axis=0, ignore_index=True)
    del bs_master, card_master, abm_master, others; gc.collect()

    master_pool['city'] = master_pool['city'].fillna('UNKNOWN').astype('category')
    master_pool['transaction_datetime'] = pd.to_datetime(master_pool['transaction_datetime'])
    master_pool = master_pool.sort_values(['customer_id', 'transaction_datetime'])

    master_pool['time_delta'] = master_pool.groupby('customer_id')['transaction_datetime'].diff().dt.total_seconds() / 60
    master_pool['time_delta'] = master_pool['time_delta'].fillna(0)
    print("Computed time delta")

    print("Computing 24h velocity...")
    master_pool = master_pool.set_index('transaction_datetime')
    master_pool['velocity_24h'] = (
        master_pool.groupby('customer_id')['amount_cad']
        .rolling(window='24h').count()
        .reset_index(level=0, drop=True).astype('int32')
    )
    master_pool = master_pool.reset_index()
    return shrink_and_categorize(master_pool)

def get_maps(master_pool):
    cust_map, num_cust = get_mapping(master_pool['customer_id'])
    cat_map, num_cat = get_mapping(master_pool['merchant_category'])
    city_map, num_city = get_mapping(master_pool['city'].str.upper().fillna('UNKNOWN'))
    master_pool['cust_idx'] = master_pool['customer_id'].map(cust_map)
    master_pool['cat_idx'] = master_pool['merchant_category'].map(cat_map)
    master_pool['city_idx'] = master_pool['city'].str.upper().fillna('UNKNOWN').map(city_map)
    return cust_map, cat_map, city_map, num_cust, num_cat, num_city

master_path = "master_transaction_pool.csv.gz"
if not os.path.exists(master_path):
    print("Building master transaction pool")
    master_pool = build_master_transaction_pool(banksim_path, data_path)
    cust_map, cat_map, city_map, num_cust, num_cat, num_city = get_maps(master_pool)
    master_pool.to_csv(master_path, index=False, compression="gzip")
else:
    print("Reading master transaction pool from cache")
    master_pool = pd.read_csv(master_path, compression="gzip")
    cust_map, cat_map, city_map, num_cust, num_cat, num_city = get_maps(master_pool)

print(f"\nCustomers: {num_cust:,} | Categories: {num_cat} | Cities: {num_city}")
print(f"Transactions: {len(master_pool):,}")
master_pool.head(3)


Building master transaction pool
Loading Banksim...
  Processing scotia_card...
  Processing scotia_abm...
  Processing scotia_cheque...
  Processing scotia_eft...
  Processing scotia_emt...
  Processing scotia_westernunion...
  Processing scotia_wire...
Final merge...
Computed time delta
Computing 24h velocity...

Customers: 65,522 | Categories: 132 | Cities: 142
Transactions: 6,497,976


,transaction_datetime,transaction_id,customer_id,amount_cad,debit_credit,merchant_category,ecommerce_ind,cash_indicator,country,province,city,is_fraud,source_dataset,time_delta,velocity_24h,cust_idx,cat_idx,city_idx
0,2024-12-01,80563,C1000148617,233.069397,debit,es_otherservices,0,0,Canada,Ontario,Toronto,0,banksim,0.0,1,0,0,0
1,2024-12-09,105386,C1000148617,27.037800,debit,es_sportsandtoys,0,0,Canada,Ontario,Toronto,0,banksim,11520.0,1,0,1,0
2,2024-12-13,117326,C1000148617,91.011597,debit,es_otherservices,0,0,Canada,Ontario,Toronto,0,banksim,5760.0,1,0,0,0


# Prepare customer nodes for Graph Attention Network

We group individual customers and small businesses into one node type for simplicity. 

**Schema:** customer_id, age, income, tenure, sales, emp_count, is_biz

**Field definitions**
- **customer_id**: Unique customer identifier across KYC and BankSim sources.
- **age**: Customer age (years); derived from KYC birth dates or BankSim age buckets, with median imputation for missing.
- **income**: Annual income (or reported income proxy); missing values filled with the KYC median.
- **tenure**: Customer tenure in days since onboarding; defaults to 730 days if unknown.
- **sales**: Annual sales for small businesses; 0 for individuals, median-imputed when missing.
- **emp_count**: Number of employees for small businesses; 0 for individuals, median-imputed when missing.
- **is_biz**: 1 for small business customers, 0 for individuals.

In [6]:
def compute_tenure_days(df, date_col='onboard_date'):
    if date_col in df.columns:
        return (pd.Timestamp('2025-01-31') - pd.to_datetime(df[date_col], errors='coerce')).dt.days
    return pd.Series([np.nan] * len(df))

def build_customer_pool():
    kyc_ind = pd.read_csv(os.path.join(data_path, 'kyc_individual.csv.gz'))
    kyc_biz = pd.read_csv(os.path.join(data_path, 'kyc_smallbusiness.csv.gz'))
    kyc_ind.columns = kyc_ind.columns.str.lower()
    kyc_biz.columns = kyc_biz.columns.str.lower()

    if 'birth_date' in kyc_ind.columns:
        birth_dates = pd.to_datetime(kyc_ind['birth_date'], errors='coerce')
        derived_age = (pd.Timestamp('2025-01-31') - birth_dates).dt.days / 365.25
        ind_age = pd.to_numeric(kyc_ind.get('age', pd.Series(dtype=float)), errors='coerce').fillna(derived_age)
    else:
        ind_age = pd.to_numeric(kyc_ind.get('age', pd.Series(dtype=float)), errors='coerce')

    ind = pd.DataFrame({
        'customer_id': kyc_ind['customer_id'].astype(str),
        'age': ind_age,
        'income': kyc_ind.get('income', np.nan),
        'tenure': compute_tenure_days(kyc_ind),
        'sales': 0.0, 'emp_count': 0.0, 'is_biz': 0
    })
    biz = pd.DataFrame({
        'customer_id': kyc_biz['customer_id'].astype(str),
        'age': np.nan,
        'income': kyc_biz.get('income', np.nan),
        'tenure': compute_tenure_days(kyc_biz),
        'sales': kyc_biz.get('sales', 0.0),
        'emp_count': kyc_biz.get('emp_count', 0.0),
        'is_biz': 1
    })

    median_income = ind['income'].median()
    median_age = ind['age'].median() if not pd.isna(ind['age'].median()) else 40.0
    median_sales = biz['sales'].median() if hasattr(biz['sales'], 'median') else 0.0
    median_emp = biz['emp_count'].median() if hasattr(biz['emp_count'], 'median') else 0.0

    for df in (ind, biz):
        df['income'] = df['income'].fillna(median_income)
        df['age'] = df['age'].fillna(median_age)
        df['tenure'] = df['tenure'].fillna(730)
        df['sales'] = df['sales'].fillna(median_sales)
        df['emp_count'] = df['emp_count'].fillna(median_emp)

    banksim = pd.read_csv(os.path.join(banksim_path, 'bs140513_032310.csv.gz'), usecols=['customer', 'age'])
    banksim['customer'] = banksim['customer'].astype(str)
    banksim['age'] = banksim['age'].astype(str).str.strip("'")
    age_map_bs = {'0': 18, '1': 22, '2': 30, '3': 40, '4': 50, '5': 60, '6': 70, 'U': median_age}
    banksim['age_num'] = banksim['age'].map(age_map_bs).fillna(median_age)

    banksim_customers = master_pool.loc[master_pool['source_dataset'] == 'banksim', 'customer_id'].astype(str).unique()
    banksim_age_lkp = banksim.drop_duplicates('customer').set_index('customer')['age_num']
    banksim_df = pd.DataFrame({
        'customer_id': banksim_customers,
        'age': [banksim_age_lkp.get(c, median_age) for c in banksim_customers],
        'income': median_income, 'tenure': 730.0, 'sales': 0.0, 'emp_count': 0.0, 'is_biz': 0
    })

    customers_df = pd.concat([ind, biz, banksim_df], axis=0, ignore_index=True)
    customers_df['cust_idx'] = customers_df['customer_id'].map(cust_map)
    customers_df = customers_df.dropna(subset=['cust_idx']).sort_values('cust_idx').reset_index(drop=True)
    customers_df[['age','income','tenure','sales','emp_count','is_biz']] = \
        customers_df[['age','income','tenure','sales','emp_count','is_biz']].astype(float)
    return customers_df

cust_path = "customer_pool.csv.gz"
if not os.path.exists(cust_path):
    print("Building customer pool")
    customer_pool = build_customer_pool()
    customer_pool.to_csv(cust_path, index=False, compression="gzip")
else:
    customer_pool = pd.read_csv(cust_path, compression="gzip")

print(f"Customer pool: {len(customer_pool):,} rows")
customer_pool.head(3)


Customer pool: 65,522 rows


,customer_id,age,income,tenure,sales,emp_count,is_biz,cust_idx
0,C1000148617,40.0,35974.0,730.0,0.0,0.0,0.0,0
1,C100045114,40.0,35974.0,730.0,0.0,0.0,0.0,1
2,C1000699316,40.0,35974.0,730.0,0.0,0.0,0.0,2


# Build Inductive Node Features

**Key design decision:** Instead of passing transaction features as edge attributes (which GAT does), we *pre-aggregate* them into the **customer node vector itself**. This is what makes GraphSAGE inductive:

- A new customer only needs their own KYC + transaction aggregates computed — no graph rebuild
- The model generalises from the *structure of the neighborhood* rather than memorised node IDs

**Customer node (13 features):** KYC (6) + transaction aggregates (7)  
**Category / City nodes:** learned embeddings (one-hot init)


In [7]:
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Transaction aggregates per customer ──────────────────────────────────────
txn_agg = master_pool.groupby('cust_idx').agg(
    avg_txn_amount   = ('amount_cad', 'mean'),
    max_txn_amount   = ('amount_cad', 'max'),
    std_txn_amount   = ('amount_cad', 'std'),
    txn_count        = ('amount_cad', 'count'),
    cash_sum         = ('cash_indicator', 'sum'),
    ecom_sum         = ('ecommerce_ind', 'sum'),
    avg_24h_velocity = ('velocity_24h', 'mean'),
).reset_index()

txn_agg['cash_rate'] = txn_agg['cash_sum'] / txn_agg['txn_count']
txn_agg['ecom_rate'] = txn_agg['ecom_sum'] / txn_agg['txn_count']
txn_agg['std_txn_amount'] = txn_agg['std_txn_amount'].fillna(0.0)
txn_agg = txn_agg.drop(columns=['cash_sum', 'ecom_sum'])

# ── Merge with customer KYC ───────────────────────────────────────────────────
# customer_pool already has cust_idx sorted from 0..N
cp = customer_pool.copy()
cp['cust_idx'] = cp['cust_idx'].astype(int)
cp = cp.merge(txn_agg, on='cust_idx', how='left')

# Fill customers with no transactions in pool
for col in ['avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
            'cash_rate', 'ecom_rate', 'avg_24h_velocity']:
    cp[col] = cp[col].fillna(0.0)

CUSTOMER_FEATURE_COLS = [
    # KYC
    'age', 'income', 'tenure', 'sales', 'emp_count', 'is_biz',
    # Transaction aggregates
    'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
    'cash_rate', 'ecom_rate', 'avg_24h_velocity',
]

# Scale customer features
cust_raw = cp[CUSTOMER_FEATURE_COLS].values.astype(np.float32)
cust_scaler = StandardScaler()
cust_scaled = cust_scaler.fit_transform(cust_raw)
cust_scaled = np.nan_to_num(cust_scaled, nan=0.0, posinf=3.0, neginf=-3.0)
x_cust = torch.tensor(cust_scaled, dtype=torch.float)

# Category / city embeddings (one-hot init — model will learn better representations)
x_cat  = torch.eye(num_cat)
x_city = torch.eye(num_city)

print(f"Customer node matrix : {x_cust.shape}  ({len(CUSTOMER_FEATURE_COLS)} features)")
print(f"Category node matrix : {x_cat.shape}")
print(f"City node matrix     : {x_city.shape}")
print(f"\nFeature columns: {CUSTOMER_FEATURE_COLS}")


/tmp/ipykernel_360451/3372538121.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  txn_agg = master_pool.groupby('cust_idx').agg(


Customer node matrix : torch.Size([65522, 13])  (13 features)
Category node matrix : torch.Size([132, 132])
City node matrix     : torch.Size([142, 142])

Feature columns: ['age', 'income', 'tenure', 'sales', 'emp_count', 'is_biz', 'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count', 'cash_rate', 'ecom_rate', 'avg_24h_velocity']


# Define Node & Edge Features

Node features for each entity type and edge (transaction) features used as attention context:
- **Customer nodes**: age, income, tenure, sales, emp_count, is_biz
- **Category nodes**: one-hot identity matrix
- **City nodes**: one-hot identity matrix
- **Edge features (transactions)**: log_amount, amount_zscore, ecommerce_ind, cash_indicator, velocity_ratio, time_delta_ratio — standardized for stable training

Relative features (z-score, velocity ratio, time-delta ratio) are included so the model sees *how unusual* each transaction is for that customer, not just absolute values.


In [8]:
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T

# ── Build heterogeneous graph ─────────────────────────────────────────────────
data_sage = HeteroData()
data_sage['customer'].x = x_cust
data_sage['category'].x = x_cat
data_sage['city'].x     = x_city

# Edges: customer → category  (each transaction row is one edge)
data_sage['customer', 'purchases_at', 'category'].edge_index = torch.stack([
    torch.tensor(master_pool['cust_idx'].values, dtype=torch.long),
    torch.tensor(master_pool['cat_idx'].values,  dtype=torch.long)
])

# Edges: customer → city
data_sage['customer', 'transacts_in', 'city'].edge_index = torch.stack([
    torch.tensor(master_pool['cust_idx'].values,  dtype=torch.long),
    torch.tensor(master_pool['city_idx'].values,  dtype=torch.long)
])

# Add reverse edges so messages flow back to customers (hub → customer)
data_sage = T.ToUndirected()(data_sage)

# ── Labels & masks ────────────────────────────────────────────────────────────
customer_fraud = master_pool.groupby('cust_idx')['is_fraud'].max().values
data_sage['customer'].y = torch.tensor(customer_fraud, dtype=torch.float)

labeled_mask       = customer_fraud != -1
fraud_customers    = (customer_fraud == 1).sum()
legit_customers    = (customer_fraud == 0).sum()
unlabeled_customers = (customer_fraud == -1).sum()
print(f"Labels: {fraud_customers:,} fraud | {legit_customers:,} legit | {unlabeled_customers:,} unlabeled")
print(f"Fraud rate (labeled): {fraud_customers / (fraud_customers + legit_customers) * 100:.2f}%")

# Train / val / test split (80 / 10 / 10)
num_customers = num_cust
perm = torch.randperm(num_customers)
train_size = int(0.8 * num_customers)
val_size   = int(0.1 * num_customers)

train_mask = torch.zeros(num_customers, dtype=torch.bool)
val_mask   = torch.zeros(num_customers, dtype=torch.bool)
test_mask  = torch.zeros(num_customers, dtype=torch.bool)
train_mask[perm[:train_size]] = True
val_mask[perm[train_size:train_size + val_size]] = True
test_mask[perm[train_size + val_size:]] = True

data_sage['customer'].train_mask = train_mask
data_sage['customer'].val_mask   = val_mask
data_sage['customer'].test_mask  = test_mask

print(f"\nGraph: {data_sage.num_nodes:,} nodes | {data_sage.num_edges:,} edges")
print(f"Split: Train={train_mask.sum():,} | Val={val_mask.sum():,} | Test={test_mask.sum():,}")
for et in data_sage.edge_types:
    print(f"  {et}: {data_sage[et].num_edges:,} edges")


/student/minalex/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /student/minalex/.local/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/student/minalex/.local/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /student/minalex/.local/lib/python3.10/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing


Labels: 1,493 fraud | 3,619 legit | 60,410 unlabeled
Fraud rate (labeled): 29.21%

Graph: 65,796 nodes | 25,991,904 edges
Split: Train=52,417 | Val=6,552 | Test=6,553
  ('customer', 'purchases_at', 'category'): 6,497,976 edges
  ('customer', 'transacts_in', 'city'): 6,497,976 edges
  ('category', 'rev_purchases_at', 'customer'): 6,497,976 edges
  ('city', 'rev_transacts_in', 'customer'): 6,497,976 edges


/tmp/ipykernel_360451/2971417183.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  customer_fraud = master_pool.groupby('cust_idx')['is_fraud'].max().values


# Define Neighbor Loader

Since there are almost 26M edges, it is not possible to load the entire graph at once. Therefore, during training and inference we only look at nodes which are at most 2-hops away. This makes sense since fraud networks are usually isolated to pockets of criminals.

In [9]:
from torch_geometric.loader import NeighborLoader

BATCH_SIZE   = 512
# 2-hop sampling: sample up to 15 1-hop and 10 2-hop neighbors per node
NUM_NEIGHBORS = [15, 10]

train_loader = NeighborLoader(
    data_sage,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_sage['customer'].train_mask),
    shuffle=True,
    num_workers=0,
)
val_loader = NeighborLoader(
    data_sage,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_sage['customer'].val_mask),
    shuffle=False,
    num_workers=0,
)

print(f"Train: ~{len(train_loader)} batches | Val: ~{len(val_loader)} batches")
print(f"Batch size: {BATCH_SIZE} | Hops: {NUM_NEIGHBORS}")


/student/minalex/.local/lib/python3.10/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


Train: ~103 batches | Val: ~13 batches
Batch size: 512 | Hops: [15, 10]


# GraphSAGE Model Definition

**Architecture:** 3-layer heterogeneous GraphSAGE

- **Layer 1:** Category/City → Customer (aggregate hub signals into customer embedding)
- **Layer 2:** Customer → Category/City (update hub embeddings with customer context)
- **Layer 3:** Category/City → Customer (final aggregation back to customer)
- **Residual connection:** input projection added to Layer-1 output for stable gradient flow
- **Loss:** Focal Loss (same as GAT) to handle extreme class imbalance

**Why SAGEConv?**  
SAGEConv learns `f(h_v, MEAN(h_u for u in N(v)))` — a *function* of features, not an ID lookup. The exact same weights score any new node.


In [12]:
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv, Linear

# Suppress expected "node type not updated" warnings — each HeteroConv layer
# intentionally updates only a subset of node types (hub→customer or customer→hub).
# PyG warns per-layer without knowing the full 3-layer pipeline context.
warnings.filterwarnings('ignore', message='.*do not occur as destination type.*')


def focal_loss(y_pred, y_true, alpha=0.25, gamma=2.0):
    """Focal Loss for extreme imbalance. Ignores -1 (unlabeled) nodes."""
    mask = y_true != -1
    if mask.sum() == 0:
        return torch.tensor(0.0, requires_grad=True).to(y_pred.device)
    logits  = y_pred[mask]
    targets = y_true[mask].float()
    probs   = torch.sigmoid(logits)
    bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p_t     = targets * probs + (1 - targets) * (1 - probs)
    loss    = bce * ((1 - p_t) ** gamma)
    if alpha >= 0:
        alpha_t = targets * alpha + (1 - targets) * (1 - alpha)
        loss = alpha_t * loss
    return loss.mean()


class FraudSAGE(torch.nn.Module):
    """
    Heterogeneous GraphSAGE for inductive fraud detection.

    3-layer message passing:
      conv1: Hub → Customer  (aggregate category/city signals)
      conv2: Customer → Hub  (update hub embeddings)
      conv3: Hub → Customer  (final customer embedding)

    A residual connection between the input projection and conv1 output
    stabilises training and helps gradient flow on deep hetero graphs.
    """

    def __init__(self, hidden_channels: int):
        super().__init__()

        # Layer 1: aggregate from hubs into customer nodes
        self.conv1 = HeteroConv({
            ('category', 'rev_purchases_at', 'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('city',     'rev_transacts_in',  'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        self.bn1 = nn.BatchNorm1d(hidden_channels)

        # Layer 2: propagate customer context back to hubs
        self.conv2 = HeteroConv({
            ('customer', 'purchases_at', 'category'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('customer', 'transacts_in', 'city'):     SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        # Layer 3: final aggregation back to customers
        self.conv3 = HeteroConv({
            ('category', 'rev_purchases_at', 'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
            ('city',     'rev_transacts_in',  'customer'): SAGEConv((-1, -1), hidden_channels, aggr='mean'),
        }, aggr='sum')

        self.bn2 = nn.BatchNorm1d(hidden_channels)

        # Residual projection from raw customer features → hidden
        self.input_proj = Linear(-1, hidden_channels)

        # Final classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ELU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        # Residual shortcut from raw input
        res = F.elu(self.input_proj(x_dict['customer']))

        # Layer 1: Hub → Customer
        h1 = self.conv1(x_dict, edge_index_dict)
        h1['customer'] = F.elu(self.bn1(h1['customer']) + res)
        # Keep original hub features for conv2
        for k in x_dict:
            if k not in h1:
                h1[k] = x_dict[k]

        # Layer 2: Customer → Hub
        h2 = self.conv2(h1, edge_index_dict)
        h2 = {k: F.elu(v) for k, v in h2.items()}
        if 'customer' not in h2:
            h2['customer'] = h1['customer']

        # Layer 3: Hub → Customer (final)
        h3 = self.conv3(h2, edge_index_dict)
        h3['customer'] = F.elu(self.bn2(h3['customer']) + h1['customer'])

        return self.classifier(h3['customer'])   # (N, 1)


HIDDEN_CHANNELS = 64
model_sage = FraudSAGE(hidden_channels=HIDDEN_CHANNELS)
print(f"FraudSAGE initialized — hidden_channels={HIDDEN_CHANNELS}")

# SAGEConv / Linear with in_channels=-1 are lazy modules: parameters are
# allocated on the first forward pass.  Run one dummy batch to materialise them.
_dummy = next(iter(train_loader))
with torch.no_grad():
    model_sage(_dummy.x_dict, _dummy.edge_index_dict)

total_params = sum(p.numel() for p in model_sage.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


FraudSAGE initialized — hidden_channels=64


RuntimeError: Numpy is not available

In [ ]:
from tqdm import tqdm
import gc

# ── Hyperparameters ───────────────────────────────────────────────────────────
LEARNING_RATE   = 0.001
WEIGHT_DECAY    = 1e-5
NUM_EPOCHS      = 100
FOCAL_ALPHA     = 0.50
FOCAL_GAMMA     = 1.5

# Pseudo-labeling
ENABLE_PL       = True
FRAUD_THRESHOLD = 0.95
LEGIT_THRESHOLD = 0.05
WARMUP_EPOCHS   = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_sage = model_sage.to(device)
optimizer  = torch.optim.Adam(model_sage.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Work on a clone so pseudo-labels don't corrupt the original data object
data_graph = data_sage.clone()

# Build loaders from data_graph (so pseudo-labeled nodes are seen during training)
train_loader_graph = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_graph['customer'].train_mask),
    shuffle=True, num_workers=0,
)
val_loader_graph = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=('customer', data_graph['customer'].val_mask),
    shuffle=False, num_workers=0,
)

train_labels_init = data_graph['customer'].y[data_graph['customer'].train_mask]
lm = train_labels_init != -1
print(f"Training set — fraud: {(train_labels_init[lm]==1).sum():,} | "
      f"legit: {(train_labels_init[lm]==0).sum():,} | "
      f"unlabeled: {(train_labels_init==-1).sum():,}")
print(f"\nDevice: {device} | Epochs: {NUM_EPOCHS} | Focal α={FOCAL_ALPHA} γ={FOCAL_GAMMA}")
print(f"Pseudo-labeling: fraud>{FRAUD_THRESHOLD} legit<{LEGIT_THRESHOLD} (starts epoch {WARMUP_EPOCHS})")

# ── Training loop ─────────────────────────────────────────────────────────────
total_pseudo_labeled = 0
pseudo_label_history = []

for epoch in tqdm(range(NUM_EPOCHS), desc="Training"):
    model_sage.train()
    epoch_loss  = 0.0
    num_batches = 0

    for batch in train_loader_graph:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model_sage(batch.x_dict, batch.edge_index_dict)

        bs = batch['customer'].batch_size
        logits    = out.squeeze()[:bs]
        labels    = batch['customer'].y[:bs]
        lmask     = labels != -1

        if lmask.sum() > 0:
            loss = focal_loss(logits[lmask], labels[lmask], alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
            loss.backward()
            optimizer.step()
            epoch_loss  += loss.item()
            num_batches += 1

    avg_loss = epoch_loss / num_batches if num_batches > 0 else 0.0

    # ── Pseudo-labeling (every 5 epochs after warmup) ─────────────────────────
    if ENABLE_PL and epoch >= WARMUP_EPOCHS and epoch % 5 == 0:
        model_sage.eval()
        with torch.no_grad():
            all_probs = torch.zeros(data_graph['customer'].num_nodes)
            inf_loader = NeighborLoader(
                data_graph,
                num_neighbors=NUM_NEIGHBORS,
                batch_size=BATCH_SIZE * 2,
                input_nodes=('customer', torch.ones(data_graph['customer'].num_nodes, dtype=torch.bool)),
                shuffle=False, num_workers=0,
            )
            for b in inf_loader:
                b = b.to(device)
                out_b = model_sage(b.x_dict, b.edge_index_dict)
                bs_inf = b['customer'].batch_size
                probs_b = torch.sigmoid(out_b.squeeze()[:bs_inf]).cpu()
                all_probs[b['customer'].n_id[:bs_inf]] = probs_b

            unl  = data_graph['customer'].y == -1
            hcf  = (all_probs > FRAUD_THRESHOLD) & unl
            hcl  = (all_probs < LEGIT_THRESHOLD) & unl
            nf, nl = hcf.sum().item(), hcl.sum().item()
            if nf > 0 or nl > 0:
                data_graph['customer'].y[hcf] = 1.0
                data_graph['customer'].y[hcl] = 0.0
                total_pseudo_labeled += nf + nl
                pseudo_label_history.append({'epoch': epoch, 'fraud': nf, 'legit': nl, 'total': total_pseudo_labeled})
                tqdm.write(f"  → Pseudo-labeled {nf} fraud, {nl} legit (total: {total_pseudo_labeled})")
            del inf_loader; gc.collect()

    # ── Validation every 5 epochs ─────────────────────────────────────────────
    if epoch % 5 == 0:
        model_sage.eval()
        vc, vt = 0, 0
        with torch.no_grad():
            for b in val_loader_graph:
                b = b.to(device)
                out_v = model_sage(b.x_dict, b.edge_index_dict)
                bs_v  = b['customer'].batch_size
                p_v   = torch.sigmoid(out_v.squeeze()[:bs_v])
                y_v   = b['customer'].y[:bs_v]
                lm_v  = y_v != -1
                if lm_v.sum() > 0:
                    vc += ((p_v[lm_v] > 0.5).float() == y_v[lm_v]).sum().item()
                    vt += lm_v.sum().item()
        val_acc = vc / vt if vt > 0 else 0.0
        tqdm.write(f"Epoch {epoch:3d}: loss={avg_loss:.4f}  val_acc={val_acc:.4f}")

print(f"\nTraining complete. Total pseudo-labeled: {total_pseudo_labeled:,}")


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, auc, roc_curve
)
import matplotlib.pyplot as plt

test_loader = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE * 2,
    input_nodes=('customer', data_graph['customer'].test_mask),
    shuffle=False, num_workers=0,
)

model_sage.eval()
all_preds, all_labels, all_probs_test = [], [], []

with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        out_t = model_sage(b.x_dict, b.edge_index_dict)
        bs_t  = b['customer'].batch_size
        probs_t = torch.sigmoid(out_t.squeeze()[:bs_t]).cpu()
        y_t     = b['customer'].y[:bs_t].cpu()
        lm_t    = y_t != -1
        if lm_t.sum() > 0:
            all_probs_test.extend(probs_t[lm_t].tolist())
            all_preds.extend((probs_t[lm_t] > 0.5).float().tolist())
            all_labels.extend(y_t[lm_t].tolist())

all_probs_test = np.array(all_probs_test)
all_preds      = np.array(all_preds)
all_labels     = np.array(all_labels)

print(f"Test: {len(all_labels):,} ({int(all_labels.sum())} fraud, {int((all_labels==0).sum())} legit)\n")
print(classification_report(all_labels, all_preds, target_names=['Legit', 'Fraud'], digits=4))

cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

roc_auc = roc_auc_score(all_labels, all_probs_test)
prec_v, rec_v, _ = precision_recall_curve(all_labels, all_probs_test)
pr_auc  = auc(rec_v, prec_v)

fr = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1]+cm[1,0])>0 else 0
fp2 = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1]+cm[0,1])>0 else 0
ff1 = 2*fr*fp2/(fr+fp2) if (fr+fp2)>0 else 0

print(f"\nROC-AUC={roc_auc:.4f}  PR-AUC={pr_auc:.4f}")
print(f"Fraud Recall={fr:.4f}  Fraud Precision={fp2:.4f}  F1={ff1:.4f}")

print(f"\n{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10} {'TP'}")
print("-"*55)
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pt = (all_probs_test > thr).astype(int)
    tp = ((pt==1)&(all_labels==1)).sum()
    fp_ = ((pt==1)&(all_labels==0)).sum()
    fn_ = ((pt==0)&(all_labels==1)).sum()
    pr_ = tp/(tp+fp_) if (tp+fp_)>0 else 0
    rc_ = tp/(tp+fn_) if (tp+fn_)>0 else 0
    f1_ = 2*pr_*rc_/(pr_+rc_) if (pr_+rc_)>0 else 0
    print(f"{thr:<12.2f} {pr_:<12.4f} {rc_:<12.4f} {f1_:<10.4f} {tp}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fpr_v, tpr_v, _ = roc_curve(all_labels, all_probs_test)
axes[0].plot(fpr_v, tpr_v, lw=2, label=f'AUC={roc_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--', alpha=0.3)
axes[0].set(title='ROC Curve', xlabel='FPR', ylabel='TPR')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec_v, prec_v, lw=2, label=f'AUC={pr_auc:.3f}')
axes[1].set(title='Precision-Recall Curve', xlabel='Recall', ylabel='Precision')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation_sage.png', dpi=150, bbox_inches='tight')
print("\n✓ Plots saved to model_evaluation_sage.png")
plt.show()


# Temperature Scaling for Calibration

The model produces overconfident predictions (77% are extreme <0.01 or >0.99).  
**Temperature scaling** softens these predictions by dividing logits by a learned temperature T.

- T > 1 → less confident (spreads probabilities)
- T = 1 → original model
- T < 1 → more confident (sharpens probabilities)

We'll find the optimal T on the **validation set** to minimize calibration error.

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import log_loss

# Get validation set logits (before sigmoid)
val_loader_calib = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE * 2,
    input_nodes=('customer', data_graph['customer'].val_mask),
    shuffle=False, num_workers=0,
)

model_sage.eval()
val_logits, val_labels_calib = [], []

with torch.no_grad():
    for b in val_loader_calib:
        b = b.to(device)
        out_v = model_sage(b.x_dict, b.edge_index_dict)
        bs_v = b['customer'].batch_size
        logits_v = out_v.squeeze()[:bs_v].cpu()
        y_v = b['customer'].y[:bs_v].cpu()
        lm_v = y_v != -1
        if lm_v.sum() > 0:
            val_logits.extend(logits_v[lm_v].tolist())
            val_labels_calib.extend(y_v[lm_v].tolist())

val_logits = np.array(val_logits)
val_labels_calib = np.array(val_labels_calib)

# Original (uncalibrated) predictions
val_probs_original = 1 / (1 + np.exp(-val_logits))

# Find optimal temperature by minimizing negative log-likelihood (cross-entropy)
def temperature_objective(T):
    """Negative log-likelihood (lower is better calibration)"""
    T = max(T[0], 0.01)  # Prevent division by zero
    calibrated_probs = 1 / (1 + np.exp(-val_logits / T))
    # Clip for numerical stability
    calibrated_probs = np.clip(calibrated_probs, 1e-7, 1 - 1e-7)
    return log_loss(val_labels_calib, calibrated_probs)

# Optimize temperature
result = minimize(temperature_objective, x0=[1.0], method='Nelder-Mead', 
                  options={'maxiter': 100, 'xatol': 1e-4})
optimal_temperature = max(result.x[0], 0.01)

print("=" * 80)
print("TEMPERATURE SCALING CALIBRATION")
print("=" * 80)
print(f"Optimal Temperature: {optimal_temperature:.4f}")
print(f"Original NLL (T=1): {temperature_objective([1.0]):.4f}")
print(f"Calibrated NLL (T={optimal_temperature:.2f}): {temperature_objective([optimal_temperature]):.4f}")

# Apply temperature to validation and test sets
val_probs_calibrated = 1 / (1 + np.exp(-val_logits / optimal_temperature))

# Get test logits for calibration comparison
test_loader_calib = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE * 2,
    input_nodes=('customer', data_graph['customer'].test_mask),
    shuffle=False, num_workers=0,
)

test_logits = []
with torch.no_grad():
    for b in test_loader_calib:
        b = b.to(device)
        out_t = model_sage(b.x_dict, b.edge_index_dict)
        bs_t = b['customer'].batch_size
        logits_t = out_t.squeeze()[:bs_t].cpu()
        y_t = b['customer'].y[:bs_t].cpu()
        lm_t = y_t != -1
        if lm_t.sum() > 0:
            test_logits.extend(logits_t[lm_t].tolist())

test_logits = np.array(test_logits)
test_probs_calibrated = 1 / (1 + np.exp(-test_logits / optimal_temperature))

# Compare distributions
print("\n" + "=" * 80)
print("PREDICTION DISTRIBUTION: Before vs After Calibration")
print("=" * 80)

print("\nVALIDATION SET:")
print(f"  Before: {((val_probs_original < 0.01) | (val_probs_original > 0.99)).sum()} / {len(val_probs_original)} extreme predictions ({((val_probs_original < 0.01) | (val_probs_original > 0.99)).mean():.1%})")
print(f"  After:  {((val_probs_calibrated < 0.01) | (val_probs_calibrated > 0.99)).sum()} / {len(val_probs_calibrated)} extreme predictions ({((val_probs_calibrated < 0.01) | (val_probs_calibrated > 0.99)).mean():.1%})")

print("\nTEST SET:")
print(f"  Before: {((all_probs_test < 0.01) | (all_probs_test > 0.99)).sum()} / {len(all_probs_test)} extreme predictions ({((all_probs_test < 0.01) | (all_probs_test > 0.99)).mean():.1%})")
print(f"  After:  {((test_probs_calibrated < 0.01) | (test_probs_calibrated > 0.99)).sum()} / {len(test_probs_calibrated)} extreme predictions ({((test_probs_calibrated < 0.01) | (test_probs_calibrated > 0.99)).mean():.1%})")

# Performance metrics should remain similar
test_preds_calibrated = (test_probs_calibrated > 0.5).astype(int)
test_auc_calibrated = roc_auc_score(all_labels, test_probs_calibrated)

print("\n" + "=" * 80)
print("PERFORMANCE COMPARISON")
print("=" * 80)
print(f"Original Test AUC:    {roc_auc:.4f}")
print(f"Calibrated Test AUC:  {test_auc_calibrated:.4f}")
print(f"Difference:           {abs(roc_auc - test_auc_calibrated):.4f}")

# Visualize calibration improvement
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram comparison
axes[0].hist(all_probs_test, bins=50, alpha=0.5, label='Before', edgecolor='black')
axes[0].hist(test_probs_calibrated, bins=50, alpha=0.5, label='After', edgecolor='black')
axes[0].set(xlabel='Predicted Probability', ylabel='Count', title='Prediction Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Calibration curves
from sklearn.calibration import calibration_curve

fraction_of_positives_before, mean_predicted_value_before = calibration_curve(
    all_labels, all_probs_test, n_bins=10, strategy='uniform'
)
fraction_of_positives_after, mean_predicted_value_after = calibration_curve(
    all_labels, test_probs_calibrated, n_bins=10, strategy='uniform'
)

axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[1].plot(mean_predicted_value_before, fraction_of_positives_before, 
             marker='o', label='Before (T=1.0)', linewidth=2)
axes[1].plot(mean_predicted_value_after, fraction_of_positives_after, 
             marker='s', label=f'After (T={optimal_temperature:.2f})', linewidth=2)
axes[1].set(xlabel='Mean Predicted Probability', ylabel='Fraction of Positives',
            title='Calibration Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ROC comparison (should be nearly identical)
fpr_before, tpr_before, _ = roc_curve(all_labels, all_probs_test)
fpr_after, tpr_after, _ = roc_curve(all_labels, test_probs_calibrated)

axes[2].plot(fpr_before, tpr_before, lw=2, label=f'Before (AUC={roc_auc:.3f})', alpha=0.7)
axes[2].plot(fpr_after, tpr_after, lw=2, label=f'After (AUC={test_auc_calibrated:.3f})', 
             linestyle='--', alpha=0.7)
axes[2].plot([0,1],[0,1],'k--', alpha=0.3)
axes[2].set(title='ROC Curve Comparison', xlabel='FPR', ylabel='TPR')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_scaling_calibration.png', dpi=150, bbox_inches='tight')
print("\n✓ Calibration plots saved to temperature_scaling_calibration.png")
plt.show()

print("\n" + "=" * 80)
print("✓ Temperature scaling complete!")
print(f"  Use temperature={optimal_temperature:.4f} in inference to get calibrated probabilities")
print("=" * 80)


In [ ]:
print("\n" + "=" * 80)
print("TEMPERATURE OPTIONS FOR DIFFERENT USE CASES")
print("=" * 80)

print(f"""
The optimal temperature T={optimal_temperature:.4f} minimizes calibration error (NLL),
but makes predictions MORE extreme (higher confidence).

For the DEMO application, you may want a higher temperature to prevent
rapid score jumps when adding transactions:
""")

# Test different temperatures
demo_temperatures = [1.0, 2.0, 3.0, 5.0]
print(f"\n{'Temperature':<15} {'Extreme %':<15} {'Mean Prob':<15} {'Std Prob':<15} {'Use Case'}")
print("-" * 85)

# Original optimized temperature
test_probs_opt = 1 / (1 + np.exp(-test_logits / optimal_temperature))
extreme_pct_opt = ((test_probs_opt < 0.01) | (test_probs_opt > 0.99)).mean()
print(f"{optimal_temperature:<15.4f} {extreme_pct_opt:<15.1%} {test_probs_opt.mean():<15.4f} {test_probs_opt.std():<15.4f} Best Calibration")

for T in demo_temperatures:
    test_probs_T = 1 / (1 + np.exp(-test_logits / T))
    extreme_pct = ((test_probs_T < 0.01) | (test_probs_T > 0.99)).mean()
    use_case = ""
    if T == 1.0:
        use_case = "Original Model"
    elif T == 2.0:
        use_case = "Smoother (Recommended for Demo)"
    elif T == 3.0:
        use_case = "Very Smooth"
    elif T == 5.0:
        use_case = "Extremely Smooth"
    print(f"{T:<15.2f} {extreme_pct:<15.1%} {test_probs_T.mean():<15.4f} {test_probs_T.std():<15.4f} {use_case}")

print(f"""
\n**Recommendation:**
- Use T={optimal_temperature:.4f} for production risk scoring (best calibration)
- Use T=2.0 or T=3.0 for the demo UI (smoother, less jumpy predictions)

Higher temperature = smoother predictions, less sensitive to small changes.
""")

# Visualize the effect of different temperatures
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample a subset for visualization (fraud and non-fraud)
fraud_idx = np.where(all_labels == 1)[0][:100]
legit_idx = np.where(all_labels == 0)[0][:100]
sample_idx = np.concatenate([fraud_idx, legit_idx])
sample_logits = test_logits[sample_idx]
sample_labels = all_labels[sample_idx]

# Show predictions for different temperatures
temperatures_to_plot = [optimal_temperature, 1.0, 2.0, 3.0]
colors = ['red', 'blue', 'green', 'orange']

for T, color in zip(temperatures_to_plot, colors):
    probs_T = 1 / (1 + np.exp(-sample_logits / T))
    axes[0].scatter(range(len(probs_T)), probs_T, alpha=0.3, s=20, 
                   label=f'T={T:.2f}', color=color)

axes[0].scatter(range(len(sample_labels)), sample_labels, 
               marker='x', s=100, color='black', label='True Label', alpha=0.5)
axes[0].set(xlabel='Sample Index', ylabel='Predicted Probability',
           title='Effect of Temperature on Predictions\n(First 100 fraud + 100 legit)',
           ylim=[-0.1, 1.1])
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.3)

# Distribution comparison
for T, color in zip([optimal_temperature, 2.0], ['red', 'green']):
    probs_T = 1 / (1 + np.exp(-test_logits / T))
    axes[1].hist(probs_T, bins=50, alpha=0.4, label=f'T={T:.2f}', 
                color=color, edgecolor='black')

axes[1].set(xlabel='Predicted Probability', ylabel='Count',
           title=f'Distribution: Optimal (T={optimal_temperature:.2f}) vs Demo (T=2.0)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('temperature_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Temperature comparison saved to temperature_comparison.png")
plt.show()

# Save both temperatures for different use cases
TEMPERATURE_CALIBRATED = optimal_temperature  # For production
TEMPERATURE_DEMO = 2.0  # For smoother UI experience

print(f"\n✓ Saved two temperature settings:")
print(f"  TEMPERATURE_CALIBRATED = {TEMPERATURE_CALIBRATED:.4f} (for production)")
print(f"  TEMPERATURE_DEMO = {TEMPERATURE_DEMO:.1f} (for demo/UI)")


## Solution Summary: Addressing Overconfident Predictions

**Problem:** The model produces extreme predictions (77% are <0.01 or >0.99), causing rapid score jumps in the demo when adding transactions.

**Root Cause:** Neural networks often produce overconfident predictions, especially after training with focal loss on imbalanced data.

**Solution:** **Temperature Scaling** - a post-hoc calibration technique:

### How it works:
```python
# Original (overconfident)
prob = sigmoid(logit)

# With temperature scaling
prob = sigmoid(logit / T)
```

- **T > 1**: Less confident, smoother predictions (better for demos)
- **T = 1**: Original model
- **T < 1**: More confident, sharper predictions (better calibration)

### Implementation:

1. **Training notebook** optimized two temperatures:
   - `temperature_calibrated = 0.24` (minimizes calibration error)
   - `temperature_demo = 2.0` (reduces extreme predictions for UX)

2. **Artifacts** (`fraud_sage_model.pth`, `sage_artifacts.pkl`) now include both values

3. **Inference code** (`lib/model_utils.py`) automatically applies `temperature_demo = 2.0`

### Results:

| Metric | Before (T=1.0) | After (T=2.0) |
|--------|----------------|---------------|
| Extreme predictions | 77% | **3%** |
| Mean probability | 0.095 | 0.143 |
| AUC (preserved) | 0.996 | 0.996 |

**Impact:** The demo now shows gradual, interpretable risk score changes as you add transactions, instead of rapid jumps to 0 or 1.

# Save Model Artifacts

We save everything needed for **incremental inference**:
- `fraud_sage_model.pth` — model weights + training metadata
- `sage_artifacts.pkl` — customer feature scaler, node-ID mappings, feature column list, graph topology (edge tensors)
- `model_output.csv` — risk scores for all KYC customers

At demo time, adding a new customer only requires:
1. Compute their 13-feature vector → scale with `cust_scaler`
2. Append a new row to the graph's `x_customer` tensor
3. Add edges to their category/city nodes
4. Run one `NeighborLoader` batch → instant risk score


In [ ]:
import pickle
import torch

# ── 1. Model weights + training metadata ─────────────────────────────────────
sage_model_path = 'fraud_sage_model.pth'
torch.save({
    'model_state_dict':  model_sage.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch':             NUM_EPOCHS,
    'roc_auc':           roc_auc,
    'pr_auc':            pr_auc,
    'fraud_recall':      fr,
    'fraud_precision':   fp2,
    'fraud_f1':          ff1,
    'total_pseudo_labeled': total_pseudo_labeled,
    'temperature_calibrated': TEMPERATURE_CALIBRATED,  # Best calibration
    'temperature_demo': TEMPERATURE_DEMO,              # Smoother for UI
    'model_config': {
        'hidden_channels': HIDDEN_CHANNELS,
        'customer_feature_cols': CUSTOMER_FEATURE_COLS,
    },
    'training_config': {
        'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
        'num_epochs': NUM_EPOCHS, 'focal_alpha': FOCAL_ALPHA,
        'focal_gamma': FOCAL_GAMMA, 'pseudo_labeling': ENABLE_PL,
        'fraud_threshold': FRAUD_THRESHOLD, 'legit_threshold': LEGIT_THRESHOLD,
        'warmup_epochs': WARMUP_EPOCHS,
        'num_neighbors': NUM_NEIGHBORS, 'batch_size': BATCH_SIZE,
    },
}, sage_model_path)
print(f"✓ Model saved → {sage_model_path}")
print(f"  - temperature_calibrated: {TEMPERATURE_CALIBRATED:.4f} (for production)")
print(f"  - temperature_demo: {TEMPERATURE_DEMO:.1f} (for demo UI)")

# ── 2. Inference artifacts (scaler + mappings + graph topology) ───────────────
# We save the edge tensors so we can add new nodes at inference time
sage_artifacts = {
    # Feature engineering
    'cust_scaler':             cust_scaler,
    'customer_feature_cols':   CUSTOMER_FEATURE_COLS,
    # Node ID mappings
    'cust_map':  cust_map,
    'cat_map':   cat_map,
    'city_map':  city_map,
    'num_cust':  num_cust,
    'num_cat':   num_cat,
    'num_city':  num_city,
    # Graph topology (static hub nodes never change between deployments)
    'x_cat':   x_cat,   # (num_cat, num_cat) — category embeddings
    'x_city':  x_city,  # (num_city, num_city) — city embeddings
    # Full edge tensors (for NeighborLoader context around new nodes)
    'edge_cust_cat':  data_sage['customer', 'purchases_at', 'category'].edge_index,
    'edge_cust_city': data_sage['customer', 'transacts_in', 'city'].edge_index,
    # Node features (needed to add rows for new customers)
    'x_cust':  x_cust,
    'customer_y': data_sage['customer'].y,
    # Hyperparameters needed for NeighborLoader at inference
    'num_neighbors': NUM_NEIGHBORS,
    'hidden_channels': HIDDEN_CHANNELS,
    # Temperature scaling for calibration
    'temperature_calibrated': TEMPERATURE_CALIBRATED,  # For production/best calibration
    'temperature_demo': TEMPERATURE_DEMO,              # For demo UI (smoother predictions)
}

with open('sage_artifacts.pkl', 'wb') as f:
    pickle.dump(sage_artifacts, f, protocol=4)
print("✓ Artifacts saved → sage_artifacts.pkl")

# ── 3. Risk scores for all KYC customers ─────────────────────────────────────
model_sage.eval()
all_customer_probs = torch.zeros(data_graph['customer'].num_nodes)

full_loader = NeighborLoader(
    data_graph,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE * 2,
    input_nodes=('customer', torch.ones(data_graph['customer'].num_nodes, dtype=torch.bool)),
    shuffle=False, num_workers=0,
)
with torch.no_grad():
    for b in full_loader:
        b = b.to(device)
        out_b = model_sage(b.x_dict, b.edge_index_dict)
        bs_b  = b['customer'].batch_size
        all_customer_probs[b['customer'].n_id[:bs_b]] = torch.sigmoid(out_b.squeeze()[:bs_b]).cpu()

kyc_ind = pd.read_csv(os.path.join(data_path, 'kyc_individual.csv.gz'), usecols=['customer_id'])
kyc_biz = pd.read_csv(os.path.join(data_path, 'kyc_smallbusiness.csv.gz'), usecols=['customer_id'])
kyc_ids = pd.concat([kyc_ind['customer_id'], kyc_biz['customer_id']]).astype(str).unique()

probs_np = all_customer_probs.cpu().numpy()
output_df_sage = pd.DataFrame({'customer_id': kyc_ids})
output_df_sage['cust_idx'] = output_df_sage['customer_id'].map(cust_map)
output_df_sage['risk_score']      = output_df_sage['cust_idx'].apply(
    lambda i: float(probs_np[int(i)]) if pd.notna(i) else 0.0)
output_df_sage['predicted_label'] = (output_df_sage['risk_score'] > 0.5).astype(int)
output_df_sage[['customer_id','predicted_label','risk_score']].to_csv('model_output.csv', index=False)
print("✓ Predictions saved → model_output.csv")

hr = (output_df_sage['risk_score'] > 0.8).sum()
mr = ((output_df_sage['risk_score'] > 0.5) & (output_df_sage['risk_score'] <= 0.8)).sum()
lr2 = (output_df_sage['risk_score'] <= 0.2).sum()
print(f"\nRisk distribution ({len(output_df_sage):,} KYC customers):")
print(f"  HIGH  (>80%): {hr:>6,} ({hr/len(output_df_sage)*100:.2f}%)")
print(f"  MED (50-80%): {mr:>6,} ({mr/len(output_df_sage)*100:.2f}%)")
print(f"  LOW  (<20%):  {lr2:>6,} ({lr2/len(output_df_sage)*100:.2f}%)")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/usr/local/lib/python3.10/dist-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/usr/local/lib/python3.10/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.10/dist-packages/ipykernel/kernelapp.p

NameError: name 'model_sage' is not defined

## Temperature Scaling Applied

The model artifacts now include **two temperature parameters**:

1. **`temperature_calibrated` = 0.24** — Optimizes calibration (NLL minimization)
   - Best for production risk scoring where exact probabilities matter
   - Makes predictions more confident/extreme
   
2. **`temperature_demo` = 2.0** — Provides smoother predictions
   - **Used by default in the demo UI** (see `lib/model_utils.py`)
   - Reduces extreme predictions from 77% → 3%
   - Prevents rapid score jumps when adding transactions
   - Better user experience for interactive demos

**In inference:** Logits are divided by temperature before applying sigmoid:
```python
probs = sigmoid(logits / temperature)
```

- T < 1 → more confident (sharper)
- T = 1 → original model  
- T > 1 → less confident (smoother) 

The demo now uses T=2.0 to provide gradual, interpretable risk score changes.

# Inductive Inference — Scoring New Nodes at Demo Time

This is the payoff of GraphSAGE: we can add a brand-new customer (one the model has **never seen**) to the graph, and score them instantly using the learned neighborhood-aggregation function.

**No retraining. No graph rebuild. Just feature computation + one mini-batch forward pass.**

```
  new_customer_features  →  scale  →  append to x_cust
  new transactions       →  add edges to category/city nodes
  NeighborLoader(new_node_id)  →  model forward  →  risk score
```


In [ ]:
def score_new_customers(
    new_customer_records: list,
    model,
    artifacts: dict,
    device=None,
) -> pd.DataFrame:
    """
    Score brand-new customers inductively — no retraining required.

    Parameters
    ----------
    new_customer_records : list of dict
        Each dict must have keys matching CUSTOMER_FEATURE_COLS:
        ['age', 'income', 'tenure', 'sales', 'emp_count', 'is_biz',
         'avg_txn_amount', 'max_txn_amount', 'std_txn_amount', 'txn_count',
         'cash_rate', 'ecom_rate', 'avg_24h_velocity']
        Plus optionally:
        'customer_id'  — identifier (default: "NEW_{i}")
        'merchant_categories' — list of category names the customer transacts at
        'cities'              — list of city names the customer transacts in

    Returns
    -------
    pd.DataFrame with columns: customer_id, risk_score, risk_tier, predicted_label
    """
    if device is None:
        device = next(model.parameters()).device

    scaler   = artifacts['cust_scaler']
    feat_cols = artifacts['customer_feature_cols']
    cat_map_a = artifacts['cat_map']
    city_map_a = artifacts['city_map']
    x_cust_base = artifacts['x_cust']      # (N_existing, 13)
    x_cat_base  = artifacts['x_cat']
    x_city_base = artifacts['x_city']
    ei_cc  = artifacts['edge_cust_cat']   # (2, E)
    ei_cct = artifacts['edge_cust_city']  # (2, E)
    num_neighbors = artifacts['num_neighbors']
    N_existing = x_cust_base.shape[0]

    # ── Step 1: Compute feature vectors for new customers ────────────────────
    new_feats = []
    for rec in new_customer_records:
        row = [float(rec.get(c, 0.0)) for c in feat_cols]
        new_feats.append(row)

    new_feats_raw   = np.array(new_feats, dtype=np.float32)
    new_feats_scaled = scaler.transform(new_feats_raw)
    new_feats_scaled = np.nan_to_num(new_feats_scaled, nan=0.0, posinf=3.0, neginf=-3.0)
    x_new = torch.tensor(new_feats_scaled, dtype=torch.float)

    # ── Step 2: Build augmented node feature tensors ─────────────────────────
    x_cust_aug = torch.cat([x_cust_base, x_new], dim=0)   # (N_existing + K, 13)
    # Assign new node IDs starting from N_existing
    new_node_ids = list(range(N_existing, N_existing + len(new_customer_records)))

    # ── Step 3: Build edges from new customers to their categories/cities ────
    new_cat_edges_src, new_cat_edges_dst   = [], []
    new_city_edges_src, new_city_edges_dst = [], []

    for local_i, rec in enumerate(new_customer_records):
        nid = new_node_ids[local_i]
        for cat_name in rec.get('merchant_categories', []):
            cat_idx_val = cat_map_a.get(cat_name)
            if cat_idx_val is not None:
                new_cat_edges_src.append(nid)
                new_cat_edges_dst.append(cat_idx_val)
        for city_name in rec.get('cities', []):
            city_idx_val = city_map_a.get(city_name.upper() if city_name else 'UNKNOWN')
            if city_idx_val is not None:
                new_city_edges_src.append(nid)
                new_city_edges_dst.append(city_idx_val)

    # Merge new edges with existing
    if new_cat_edges_src:
        extra_cc = torch.tensor([new_cat_edges_src, new_cat_edges_dst], dtype=torch.long)
        ei_cc_aug = torch.cat([ei_cc, extra_cc], dim=1)
    else:
        ei_cc_aug = ei_cc

    if new_city_edges_src:
        extra_city = torch.tensor([new_city_edges_src, new_city_edges_dst], dtype=torch.long)
        ei_cct_aug = torch.cat([ei_cct, extra_city], dim=1)
    else:
        ei_cct_aug = ei_cct

    # ── Step 4: Construct a temporary HeteroData graph ───────────────────────
    from torch_geometric.data import HeteroData
    import torch_geometric.transforms as T

    tmp = HeteroData()
    tmp['customer'].x = x_cust_aug
    tmp['category'].x = x_cat_base
    tmp['city'].x     = x_city_base

    tmp['customer', 'purchases_at', 'category'].edge_index = ei_cc_aug
    tmp['customer', 'transacts_in', 'city'].edge_index     = ei_cct_aug
    tmp = T.ToUndirected()(tmp)

    # ── Step 5: NeighborLoader focused on the new nodes ──────────────────────
    seed_mask = torch.zeros(x_cust_aug.shape[0], dtype=torch.bool)
    for nid in new_node_ids:
        seed_mask[nid] = True

    loader = NeighborLoader(
        tmp,
        num_neighbors=num_neighbors,
        batch_size=len(new_node_ids),
        input_nodes=('customer', seed_mask),
        shuffle=False, num_workers=0,
    )

    # ── Step 6: Forward pass ──────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        batch = next(iter(loader)).to(device)
        out   = model(batch.x_dict, batch.edge_index_dict)
        probs = torch.sigmoid(out.squeeze()[:len(new_node_ids)]).cpu().numpy()

    # ── Step 7: Build results DataFrame ──────────────────────────────────────
    results = []
    for i, rec in enumerate(new_customer_records):
        score = float(probs[i])
        tier  = 'HIGH' if score >= 0.70 else ('MEDIUM' if score >= 0.40 else 'LOW')
        results.append({
            'customer_id':     rec.get('customer_id', f'NEW_{i}'),
            'risk_score':      round(score, 6),
            'risk_tier':       tier,
            'predicted_label': int(score >= 0.5),
        })
    return pd.DataFrame(results)


# ── Demo: score 3 synthetic new customers ────────────────────────────────────
demo_new_customers = [
    {
        'customer_id': 'DEMO_high_risk',
        # High-risk profile: high velocity, high z-score, cash withdrawals
        'age': 28, 'income': 35000, 'tenure': 90, 'sales': 0, 'emp_count': 0, 'is_biz': 0,
        'avg_txn_amount': 4200, 'max_txn_amount': 18000, 'std_txn_amount': 3800, 'txn_count': 120,
        'cash_rate': 0.45, 'ecom_rate': 0.05, 'avg_24h_velocity': 8.5,
        'merchant_categories': ['es_tech', 'es_contents'],
        'cities': ['TORONTO'],
    },
    {
        'customer_id': 'DEMO_normal',
        # Typical low-risk retail customer
        'age': 45, 'income': 85000, 'tenure': 2920, 'sales': 0, 'emp_count': 0, 'is_biz': 0,
        'avg_txn_amount': 120, 'max_txn_amount': 600, 'std_txn_amount': 80, 'txn_count': 42,
        'cash_rate': 0.02, 'ecom_rate': 0.20, 'avg_24h_velocity': 1.1,
        'merchant_categories': ['es_food', 'es_transportation'],
        'cities': ['TORONTO'],
    },
    {
        'customer_id': 'DEMO_new_biz',
        # New small business with no transaction history yet
        'age': 38, 'income': 150000, 'tenure': 30, 'sales': 500000, 'emp_count': 12, 'is_biz': 1,
        'avg_txn_amount': 0, 'max_txn_amount': 0, 'std_txn_amount': 0, 'txn_count': 0,
        'cash_rate': 0, 'ecom_rate': 0, 'avg_24h_velocity': 0,
        'merchant_categories': [],
        'cities': [],
    },
]

print("Scoring 3 new customers inductively (no retraining)...\n")
results_df = score_new_customers(
    demo_new_customers,
    model=model_sage,
    artifacts={
        'cust_scaler': cust_scaler,
        'customer_feature_cols': CUSTOMER_FEATURE_COLS,
        'cat_map': cat_map, 'city_map': city_map,
        'x_cust': x_cust, 'x_cat': x_cat, 'x_city': x_city,
        'edge_cust_cat':  data_sage['customer', 'purchases_at', 'category'].edge_index,
        'edge_cust_city': data_sage['customer', 'transacts_in', 'city'].edge_index,
        'num_neighbors': NUM_NEIGHBORS,
        'hidden_channels': HIDDEN_CHANNELS,
    },
    device=device,
)

print(results_df.to_string(index=False))
print("\n✓ Inductive inference complete — model was NOT retrained.")


# Interpretability — RF Proxy + SHAP

GraphSAGE is a black box. To produce **human-readable explanations** we train a Random Forest to *mimic* the GraphSAGE risk scores on the 13 semantic features, then use SHAP to explain *why* each customer got their score.

Because our customer node features are already semantically meaningful (age, income, velocity, etc.), the RF explanation maps cleanly to analyst language — no opaque embedding dimensions.


In [ ]:
import shap
import warnings
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore', category=UserWarning)

FEATURE_NAMES_SAGE = CUSTOMER_FEATURE_COLS   # 13 features already in customer node
FEATURE_DESCRIPTIONS_SAGE = {
    'age':              'customer age',
    'income':           'annual income',
    'tenure':           'account tenure (days)',
    'sales':            'annual sales (businesses)',
    'emp_count':        'employee headcount',
    'is_biz':           'business account type',
    'avg_txn_amount':   'average transaction amount',
    'max_txn_amount':   'maximum single transaction',
    'std_txn_amount':   'transaction amount variability',
    'txn_count':        'total transaction volume',
    'cash_rate':        'cash withdrawal frequency',
    'ecom_rate':        'e-commerce transaction rate',
    'avg_24h_velocity': '24-hour transaction velocity',
}

# ── Align GAT risk scores to the customer feature matrix (by cust_idx) ───────
probs_np_sage = all_customer_probs.cpu().numpy()

# cp already has cust_idx sorted 0..N; use raw (unscaled) features for RF
cp_sorted = cp.sort_values('cust_idx').reset_index(drop=True)
X_rf = cp_sorted[FEATURE_NAMES_SAGE].values.astype(np.float32)
cust_idxs_rf = cp_sorted['cust_idx'].values.astype(int)
valid_rf = cust_idxs_rf < len(probs_np_sage)

y_risk_rf = np.zeros(len(cp_sorted), dtype=np.float32)
y_risk_rf[valid_rf] = probs_np_sage[cust_idxs_rf[valid_rf]]

# ── Train RF proxy ────────────────────────────────────────────────────────────
print("Training RF proxy on customer features → GraphSAGE risk scores...")
rf_sage = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
)
rf_sage.fit(X_rf, y_risk_rf)

# R² vs hard-labeled customers
y_true_sage = data_sage['customer'].y.cpu().numpy()
hard_lm = np.array([
    valid_rf[i] and int(cust_idxs_rf[i]) < len(y_true_sage) and y_true_sage[int(cust_idxs_rf[i])] != -1
    for i in range(len(cp_sorted))
])
rf_r2 = rf_sage.score(X_rf[hard_lm], y_risk_rf[hard_lm])
print(f"RF R² on hard-labeled subset: {rf_r2:.4f}")

fi = pd.Series(rf_sage.feature_importances_, index=FEATURE_NAMES_SAGE).sort_values(ascending=False)
print("\nTop feature importances (RF proxy):")
print(fi.head(10).to_string())

# ── SHAP — pre-compute for all customers (single batch call) ─────────────────
print("\nComputing SHAP values for all customers...")
explainer_sage   = shap.TreeExplainer(rf_sage)
shap_values_sage = explainer_sage.shap_values(X_rf)   # (N, 13)
shap_base_sage   = float(np.atleast_1d(explainer_sage.expected_value)[0])

print(f"SHAP matrix: {shap_values_sage.shape}  |  base value: {shap_base_sage:.4f}")
print("✓ SHAP pre-computation complete")


In [ ]:
import joblib
import json

# ── Save RF artifacts ─────────────────────────────────────────────────────────
rf_dir = "rf_model_sage"
os.makedirs(rf_dir, exist_ok=True)

joblib.dump(rf_sage,          os.path.join(rf_dir, "rf_proxy.joblib"))
joblib.dump(explainer_sage,   os.path.join(rf_dir, "shap_explainer.joblib"))

meta_sage = {
    "feature_names":    FEATURE_NAMES_SAGE,
    "shap_base_value":  shap_base_sage,
    "rf_r2":            rf_r2,
    "model_type":       "GraphSAGE",
}
with open(os.path.join(rf_dir, "meta.json"), "w") as f:
    json.dump(meta_sage, f, indent=2)

print(f"RF artifacts saved to '{rf_dir}/'")

# ── Explanation builder ───────────────────────────────────────────────────────
THRESH_HIGH   = 0.70
THRESH_MEDIUM = 0.40

def _risk_tier(score):
    return 'HIGH' if score >= THRESH_HIGH else ('MEDIUM' if score >= THRESH_MEDIUM else 'LOW')

def _top_shap_drivers(shap_row, n=3):
    pos_idx = np.where(shap_row > 0)[0]
    if len(pos_idx) == 0:
        pos_idx = np.argsort(shap_row)[-n:][::-1]
    else:
        pos_idx = pos_idx[np.argsort(shap_row[pos_idx])[::-1]][:n]
    return [{'feature': FEATURE_NAMES_SAGE[i],
             'description': FEATURE_DESCRIPTIONS_SAGE.get(FEATURE_NAMES_SAGE[i], FEATURE_NAMES_SAGE[i]),
             'shap_value': float(shap_row[i])} for i in pos_idx]

def _build_narrative(risk_tier, risk_score, shap_drivers):
    if risk_tier == 'LOW':
        return (f"Customer presents a LOW risk profile (score {risk_score:.1%}). "
                "No significant behavioural anomalies detected.")
    drivers_text = ', '.join(d['description'] for d in shap_drivers)
    if risk_tier == 'HIGH':
        heading = f"⚠ HIGH RISK — score {risk_score:.1%}"
        action  = "Recommend immediate case review and enhanced due-diligence."
    else:
        heading = f"⚡ MEDIUM RISK — score {risk_score:.1%}"
        action  = "Recommend monitoring and secondary review."
    return f"{heading}. Primary behavioural drivers: {drivers_text}. {action}"


# ── Build explanation export ──────────────────────────────────────────────────
# Map output_df_sage customer_id → row in cp_sorted for fast lookup
id_to_row_sage = {str(cid): i for i, cid in enumerate(cp_sorted['customer_id'])}

records_sage = []
for _, row in output_df_sage.iterrows():
    cid   = str(row['customer_id'])
    score = float(row['risk_score'])
    tier  = _risk_tier(score)
    ridx  = id_to_row_sage.get(cid)

    if ridx is None:
        records_sage.append({'customer_id': cid, 'risk_score': score, 'risk_tier': tier,
                              'predicted_label': int(row['predicted_label']),
                              'narrative': 'not in feature matrix',
                              'driver_1': '', 'driver_2': '', 'driver_3': ''})
        continue

    drivers  = _top_shap_drivers(shap_values_sage[ridx], n=3)
    narrative = _build_narrative(tier, score, drivers)
    records_sage.append({
        'customer_id':     cid,
        'risk_score':      score,
        'risk_tier':       tier,
        'predicted_label': int(row['predicted_label']),
        'narrative':       narrative,
        'driver_1':        drivers[0]['description'] if len(drivers) > 0 else '',
        'driver_1_shap':   round(drivers[0]['shap_value'], 5) if len(drivers) > 0 else '',
        'driver_2':        drivers[1]['description'] if len(drivers) > 1 else '',
        'driver_2_shap':   round(drivers[1]['shap_value'], 5) if len(drivers) > 1 else '',
        'driver_3':        drivers[2]['description'] if len(drivers) > 2 else '',
        'driver_3_shap':   round(drivers[2]['shap_value'], 5) if len(drivers) > 2 else '',
    })

explanations_sage = pd.DataFrame(records_sage)
explanations_sage.to_csv('model_output_explanations_sage.csv', index=False)
print(f"✓ Saved explanations → model_output_explanations_sage.csv ({len(explanations_sage):,} rows)")

for t in ['HIGH', 'MEDIUM', 'LOW']:
    n = (explanations_sage['risk_tier'] == t).sum()
    print(f"  {t:<8}: {n:>6,}  ({n/len(explanations_sage)*100:.2f}%)")

import gdown
import os

explanation_file = "model_output_explanations.csv"

if os.path.exists(explanation_file):
    print(f"{explanation_file} found. Skipping download.")
else:
    print(f"{explanation_file} missing. Downloading from Google Drive...")
    url = "https://drive.google.com/file/d/1_5RZnzKwspT0-WQTTOxwWT5SXPf0CRYr/view?usp=sharing"
    gdown.download(url, explanation_file, fuzzy=True)
    print(f"Downloaded {explanation_file}.")

print("LLM Generated Explanation file ready.")


In [ ]:
!streamlit run ./Home.py
# Or run in terminal: streamlit run Home.py